# 🏰 AI-vs-AI Tower Defense
## Using A* Pathfinding + Minimax Algorithm + Intelligent Agents

| Component | Algorithm | Agent |
|-----------|-----------|-------|
| Route planning | A* Search | Attacker |
| Tower placement | Minimax | Defender |
| Decision loop | Perceive-Decide-Act | Both |

**How it works:**
- The **Defender Agent** uses Minimax to place towers that block the attacker as much as possible
- The **Attacker Agent** uses A* to find the shortest path around towers
- Both agents run autonomously — no human input needed


In [ ]:
# Cell 1: Imports and Setup
import heapq
import random
import time

# For visualization inside Jupyter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
import numpy as np

print("✅ All libraries imported successfully!")
print("Python version ready. Run cells top to bottom.")


In [ ]:
# Cell 2: Grid Configuration
# You can change ROWS, COLS, or wall positions here

ROWS = 12
COLS = 16

# Start (bottom-left) and Goal (top-right)
START = (ROWS - 1, 0)
GOAL  = (0, COLS - 1)

# Grid: 0 = open, 1 = wall, 2 = tower
grid = [[0]*COLS for _ in range(ROWS)]

# ── Static walls (feel free to edit these!) ────────────────────────────────────
WALLS = [
    (2, 3),(2, 4),(2, 5),(2, 6),(2, 7),
    (4, 9),(4,10),(4,11),(4,12),
    (6, 2),(6, 3),(6, 4),(6, 5),
    (8,10),(8,11),(8,12),(8,13),
    (10,5),(10,6),(10,7),
    (3,13),(3,14),
    (7, 7),(7, 8),
]
for (r, c) in WALLS:
    if (r, c) != START and (r, c) != GOAL:
        grid[r][c] = 1

print(f"✅ Grid created: {ROWS} rows × {COLS} cols")
print(f"   Start : {START}")
print(f"   Goal  : {GOAL}")
print(f"   Walls : {len(WALLS)}")


In [ ]:
# Cell 3: A* Pathfinding Algorithm
# Used by the Attacker Agent to find the shortest path

def heuristic(a, b):
    """Manhattan distance heuristic."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def in_bounds(pos):
    r, c = pos
    return 0 <= r < ROWS and 0 <= c < COLS

def astar(grid, start, goal):
    """
    A* Search Algorithm.
    Returns the shortest path as a list of (row, col) tuples,
    or None if no path exists.
    
    f(n) = g(n) + h(n)
      g(n) = cost from start to n
      h(n) = heuristic estimate from n to goal
    """
    open_set = []
    heapq.heappush(open_set, (0, start))

    came_from = {}
    g_score = {start: 0}
    f_score = {start: heuristic(start, goal)}

    while open_set:
        _, current = heapq.heappop(open_set)

        # Goal reached — reconstruct path
        if current == goal:
            path = []
            while current in came_from:
                path.append(current)
                current = came_from[current]
            path.append(start)
            return path[::-1]

        # Explore neighbours (up, down, left, right)
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            neighbor = (current[0]+dr, current[1]+dc)
            if not in_bounds(neighbor):
                continue
            if grid[neighbor[0]][neighbor[1]] != 0:  # blocked
                continue
            tentative_g = g_score[current] + 1
            if tentative_g < g_score.get(neighbor, float('inf')):
                came_from[neighbor]  = current
                g_score[neighbor]    = tentative_g
                f_score[neighbor]    = tentative_g + heuristic(neighbor, goal)
                heapq.heappush(open_set, (f_score[neighbor], neighbor))

    return None  # No path found


# Quick test
test_path = astar(grid, START, GOAL)
if test_path:
    print(f"✅ A* working! Initial path length: {len(test_path)} steps")
    print(f"   First 5 steps: {test_path[:5]}")
else:
    print("❌ No initial path found — check your wall configuration")


In [ ]:
# Cell 4: Minimax Decision Algorithm
# Used by the Defender Agent to choose where to place towers

def minimax(grid, depth, is_defender, attacker_pos):
    """
    Minimax Algorithm for tower placement.
    
    - Defender (Maximiser): wants attacker's path to be as LONG as possible
    - Attacker (Minimiser): will always find the SHORTEST path (A*)
    
    Returns: (score, best_cell_to_place_tower)
      score = path length after optimal play
      best_cell = (row, col) for tower, or None
    """
    path = astar(grid, attacker_pos, GOAL)

    # Terminal: no path or max depth reached
    if path is None:
        return float('inf'), None   # Attacker fully blocked — best for defender
    if depth == 0:
        return len(path), None

    if is_defender:
        best_score = -float('inf')
        best_cell  = None

        # Try placing a tower on each cell of the current path (except start/goal)
        candidates = path[1:-1]
        if not candidates:
            return len(path), None

        for cell in candidates:
            r, c = cell
            grid[r][c] = 2          # Simulate placing tower

            score, _ = minimax(grid, depth - 1, False, attacker_pos)

            grid[r][c] = 0          # Undo simulation

            if score > best_score:
                best_score = score
                best_cell  = cell

        return best_score, best_cell

    else:
        # Attacker responds with best A* path
        path = astar(grid, attacker_pos, GOAL)
        return (len(path) if path else float('inf')), None


# Quick test
score, tower_pos = minimax(grid, depth=2, is_defender=True, attacker_pos=START)
print(f"✅ Minimax working!")
print(f"   Best tower position: {tower_pos}")
print(f"   Expected path length after placement: {score} steps")


In [ ]:
# Cell 5: Intelligent Agent Classes
# Both agents use the Perceive → Decide → Act loop

class AttackerAgent:
    """
    Attacker Agent — uses A* to navigate to the goal.
    Perceive : reads the full grid
    Decide   : runs A* to find next step
    Act      : moves one step along the path
    """
    def __init__(self, start, goal, grid):
        self.pos      = start
        self.goal     = goal
        self.grid     = grid
        self.steps    = 0
        self.path     = []

    def perceive(self):
        """See the full grid state."""
        return self.grid

    def decide(self):
        """Run A* to find the next best move."""
        self.path = astar(self.grid, self.pos, self.goal)
        if self.path and len(self.path) > 1:
            return self.path[1]   # Next cell
        return None

    def act(self):
        """
        Move one step. Returns:
          'goal'    — reached the goal
          'moved'   — moved one step
          'blocked' — no path available
        """
        next_pos = self.decide()
        if next_pos is None:
            return 'blocked'
        self.pos    = next_pos
        self.steps += 1
        if self.pos == self.goal:
            return 'goal'
        return 'moved'

    def __repr__(self):
        return f"AttackerAgent(pos={self.pos}, steps={self.steps})"


class DefenderAgent:
    """
    Defender Agent — uses Minimax to place towers strategically.
    Perceive : reads grid + attacker position
    Decide   : runs Minimax to find best tower cell
    Act      : places tower on grid
    """
    def __init__(self, grid):
        self.grid          = grid
        self.towers_placed = []
        self.minimax_depth = 3     # ← increase for stronger AI (slower)

    def perceive(self, attacker_pos):
        """See the grid and current attacker position."""
        return self.grid, attacker_pos

    def decide(self, attacker_pos):
        """Run Minimax to find optimal tower placement."""
        _, best_cell = minimax(
            self.grid,
            depth        = self.minimax_depth,
            is_defender  = True,
            attacker_pos = attacker_pos
        )
        return best_cell

    def act(self, attacker_pos):
        """Place a tower at the Minimax-chosen position."""
        pos = self.decide(attacker_pos)
        if pos:
            r, c = pos
            self.grid[r][c] = 2   # 2 = tower
            self.towers_placed.append(pos)
            return pos
        return None

    def __repr__(self):
        return f"DefenderAgent(towers={len(self.towers_placed)})"


print("✅ Agent classes defined!")
print("   AttackerAgent  — navigates using A*")
print("   DefenderAgent  — places towers using Minimax")


In [ ]:
# Cell 6: Grid Visualizer
# Draws the current state of the map inline in Jupyter

# Color map: 0=open, 1=wall, 2=tower, 3=path, 4=attacker, 5=start, 6=goal
COLORS = {
    0: '#1E293B',   # open cell  — dark blue-gray
    1: '#374151',   # wall       — gray
    2: '#F97316',   # tower      — orange
    3: '#FDE68A',   # A* path    — yellow
    4: '#4ADE80',   # attacker   — green
    5: '#6366F1',   # start      — purple
    6: '#EC4899',   # goal       — pink
}

def draw_grid(grid, attacker_pos=None, path=None, title="Grid State", ax=None):
    """Render the grid with matplotlib."""
    show = ax is None
    if show:
        fig, ax = plt.subplots(figsize=(COLS * 0.65, ROWS * 0.65))

    # Build color matrix
    color_matrix = [[COLORS[grid[r][c]] for c in range(COLS)] for r in range(ROWS)]

    # Overlay A* path
    if path:
        for (r, c) in path[1:-1]:
            if grid[r][c] == 0:
                color_matrix[r][c] = COLORS[3]

    # Overlay start, goal, attacker
    sr, sc = START
    gr, gc = GOAL
    color_matrix[sr][sc] = COLORS[5]
    color_matrix[gr][gc] = COLORS[6]
    if attacker_pos and attacker_pos != START and attacker_pos != GOAL:
        ar, ac = attacker_pos
        color_matrix[ar][ac] = COLORS[4]

    # Draw cells
    for r in range(ROWS):
        for c in range(COLS):
            fc = color_matrix[r][c]
            rect = FancyBboxPatch((c + 0.05, ROWS - r - 0.95), 0.9, 0.9,
                                   boxstyle="round,pad=0.05",
                                   facecolor=fc, edgecolor='#0F172A', linewidth=0.5)
            ax.add_patch(rect)

            # Labels
            label = ''
            if (r, c) == START:       label = 'S'
            elif (r, c) == GOAL:      label = 'G'
            elif attacker_pos == (r,c): label = 'A'
            elif grid[r][c] == 2:     label = 'T'

            if label:
                ax.text(c + 0.5, ROWS - r - 0.5, label,
                        ha='center', va='center', fontsize=8,
                        fontweight='bold',
                        color='white' if grid[r][c] != 0 else '#111')

    ax.set_xlim(0, COLS)
    ax.set_ylim(0, ROWS)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_facecolor('#0F172A')
    ax.set_title(title, fontsize=12, fontweight='bold', color='#111827', pad=8)

    # Legend
    legend_items = [
        mpatches.Patch(color=COLORS[5], label='Start (S)'),
        mpatches.Patch(color=COLORS[6], label='Goal (G)'),
        mpatches.Patch(color=COLORS[4], label='Attacker (A)'),
        mpatches.Patch(color=COLORS[2], label='Tower (T)'),
        mpatches.Patch(color=COLORS[3], label='A* Path'),
        mpatches.Patch(color=COLORS[1], label='Wall'),
    ]
    ax.legend(handles=legend_items, loc='upper left',
              bbox_to_anchor=(1.01, 1), fontsize=8, framealpha=0.9)

    if show:
        plt.tight_layout()
        plt.show()


# Show initial grid
initial_path = astar(grid, START, GOAL)
draw_grid(grid, attacker_pos=START, path=initial_path,
          title=f"Initial Grid — A* path length: {len(initial_path)} steps")


In [ ]:
# Cell 7: Defender Phase — Place Towers Using Minimax
# The Defender Agent uses Minimax to place towers before the attacker moves

NUM_TOWERS = 5   # ← Change how many towers the defender places

attacker = AttackerAgent(START, GOAL, grid)
defender = DefenderAgent(grid)

print("=" * 50)
print("  DEFENDER PHASE — Minimax Tower Placement")
print("=" * 50)

path_before = astar(grid, START, GOAL)
print(f"Path length BEFORE towers: {len(path_before)} steps\n")

for i in range(NUM_TOWERS):
    tower_pos = defender.act(attacker.pos)
    current_path = astar(grid, START, GOAL)
    path_len = len(current_path) if current_path else float('inf')

    if tower_pos:
        print(f"Tower {i+1}: placed at {tower_pos}  |  New path length: {path_len}")
    else:
        print(f"Tower {i+1}: no beneficial placement found")

print()
path_after = astar(grid, START, GOAL)
if path_after:
    print(f"Path length AFTER towers : {len(path_after)} steps")
    print(f"Path increased by        : {len(path_after) - len(path_before)} steps")
else:
    print("Attacker is completely BLOCKED — Defender wins!")

# Visualize after defender phase
fig, axes = plt.subplots(1, 2, figsize=(COLS * 1.3, ROWS * 0.7))
draw_grid([[0 if v == 2 else v for v in row] for row in grid],
          attacker_pos=START, path=path_before,
          title="Before: Initial A* Path", ax=axes[0])
draw_grid(grid, attacker_pos=START, path=path_after,
          title=f"After: {NUM_TOWERS} Minimax Towers Placed", ax=axes[1])
plt.suptitle("Defender Phase Results", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 8: Attacker Phase — Navigate Using A*
# The Attacker Agent steps through the grid using A*, replanning each move

print("=" * 50)
print("  ATTACKER PHASE — A* Navigation")
print("=" * 50)

# Reset attacker position
attacker.pos   = START
attacker.steps = 0

history = [START]   # Track visited positions
status  = 'moving'

while True:
    result = attacker.act()
    history.append(attacker.pos)

    if result == 'goal':
        status = 'leaked'
        print(f"Step {attacker.steps:>3}: Attacker at {attacker.pos}  ← GOAL REACHED!")
        break
    elif result == 'blocked':
        status = 'blocked'
        print(f"Step {attacker.steps:>3}: BLOCKED — No path available. Defender wins!")
        break
    else:
        print(f"Step {attacker.steps:>3}: Attacker at {attacker.pos}")

    if attacker.steps > 300:
        print("Max steps reached.")
        break

print()
print(f"Result  : {'🟢 Attacker leaked to goal!' if status == 'leaked' else '🔴 Attacker blocked by defender!'}")
print(f"Steps   : {attacker.steps}")
print(f"Towers  : {len(defender.towers_placed)}")


In [ ]:
# Cell 9: Step-by-Step Replay Visualization
# Shows snapshots of the attacker's journey across the grid

# Pick evenly spaced snapshots to visualize
snap_indices = list(range(0, len(history), max(1, len(history)//6)))
if history[-1] not in [history[i] for i in snap_indices]:
    snap_indices.append(len(history)-1)
snap_indices = snap_indices[:6]  # max 6 panels

cols_viz = 3
rows_viz = (len(snap_indices) + 2) // 3
fig, axes = plt.subplots(rows_viz, cols_viz,
                          figsize=(cols_viz * COLS * 0.6, rows_viz * ROWS * 0.55))
axes = axes.flatten()

for idx, snap_i in enumerate(snap_indices):
    pos  = history[snap_i]
    path = astar(grid, pos, GOAL)
    pct  = int(snap_i / max(len(history)-1, 1) * 100)
    ttl  = f"Step {snap_i} ({pct}%)"
    if pos == GOAL: ttl += " — GOAL!"
    draw_grid(grid, attacker_pos=pos, path=path, title=ttl, ax=axes[idx])

# Hide unused axes
for i in range(len(snap_indices), len(axes)):
    axes[i].axis('off')

plt.suptitle("Attacker Journey — A* Replanning After Each Tower", 
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 10: Final Summary & Score
# Prints a summary of the full simulation

print("=" * 55)
print("  SIMULATION COMPLETE — FINAL SUMMARY")
print("=" * 55)

path_final = astar(grid, START, GOAL)
path_init  = astar([[0 if v == 2 else v for v in row] for row in grid], START, GOAL)

print(f"  Grid size          : {ROWS} x {COLS}")
print(f"  Walls placed       : {len(WALLS)}")
print(f"  Towers placed      : {len(defender.towers_placed)}")
print(f"    Positions        : {defender.towers_placed}")
print()
print(f"  Initial path length: N/A (pre-defender state)")
print(f"  Final path length  : {len(path_final) if path_final else 'BLOCKED'}")
print(f"  Attacker steps     : {attacker.steps}")
print()
if status == 'leaked':
    print("  WINNER  : 🟢 ATTACKER — reached the goal!")
else:
    print("  WINNER  : 🔴 DEFENDER — blocked all paths!")
print()
print("  AI Techniques Used:")
print("    A*       → Attacker finds shortest path each step")
print("    Minimax  → Defender picks best tower position")
print("    Agents   → Both run Perceive → Decide → Act loop")
print("=" * 55)


In [ ]:
# Cell 11: Experiment — Run a Full Fresh Simulation
# Modify parameters here and re-run everything from scratch

# ── Parameters to experiment with ─────────────────────────────────────────────
EXP_ROWS       = 10
EXP_COLS       = 14
EXP_NUM_TOWERS = 6
EXP_MINIMAX_DEPTH = 3   # Higher = smarter defender but slower

EXP_START = (EXP_ROWS - 1, 0)
EXP_GOAL  = (0, EXP_COLS - 1)

# Fresh random walls
random.seed(42)
exp_grid = [[0]*EXP_COLS for _ in range(EXP_ROWS)]
for _ in range(EXP_ROWS * EXP_COLS // 6):
    r = random.randint(1, EXP_ROWS - 2)
    c = random.randint(1, EXP_COLS - 2)
    if (r, c) != EXP_START and (r, c) != EXP_GOAL:
        exp_grid[r][c] = 1

# Patch globals temporarily
_old = (ROWS, COLS, START, GOAL)
ROWS, COLS, START, GOAL = EXP_ROWS, EXP_COLS, EXP_START, EXP_GOAL

exp_attacker = AttackerAgent(EXP_START, EXP_GOAL, exp_grid)
exp_defender = DefenderAgent(exp_grid)
exp_defender.minimax_depth = EXP_MINIMAX_DEPTH

# Defender places towers
for _ in range(EXP_NUM_TOWERS):
    exp_defender.act(exp_attacker.pos)

# Attacker navigates
exp_status = 'moving'
exp_history = [EXP_START]
while True:
    res = exp_attacker.act()
    exp_history.append(exp_attacker.pos)
    if res == 'goal':   exp_status = 'leaked';  break
    if res == 'blocked':exp_status = 'blocked'; break
    if exp_attacker.steps > 300: break

# Visualize final state
fig, ax = plt.subplots(figsize=(EXP_COLS * 0.7, EXP_ROWS * 0.7))
final_path = astar(exp_grid, exp_attacker.pos, EXP_GOAL)
winner = "Attacker wins!" if exp_status == 'leaked' else "Defender wins!"
draw_grid(exp_grid, attacker_pos=exp_attacker.pos,
          path=final_path,
          title=f"Experiment Result — {winner}  |  Steps: {exp_attacker.steps}  |  Towers: {len(exp_defender.towers_placed)}",
          ax=ax)
plt.tight_layout()
plt.show()

# Restore globals
ROWS, COLS, START, GOAL = _old
print(f"Result: {winner}")
print(f"Attacker steps: {exp_attacker.steps}, Towers placed: {len(exp_defender.towers_placed)}")
